# Step 6: Full Simulation Runner

This notebook is the **top-level orchestrator** — it wires every prior module together into a single end-to-end simulation run.

## What this notebook does

It simulates one or more years of patient flow through NYP's women's health screening program, from the moment a patient walks in the door through to treatment or loss-to-follow-up. It does this by combining two layers:

**Layer 1 — Arrivals (Sophia's queue model)**
Sophia's notebook models how patients arrive, wait, and get seen by a provider (PCP, GYN, Specialist, or ER). It handles scheduling lead times, drop-in conversions, and daily capacity constraints. We load her code unchanged via `%run` so her queue logic is reused exactly.

**Layer 2 — Screening & Follow-Up (Steps 2–5)**
Immediately after each patient is *seen* by a provider, we run:
- **Eligibility check** — is this patient due for any cancer screening today?
- **Test assignment** — which specific test modality should they receive?
- **Result draw** — stochastic result from the appropriate probability table.
- **Follow-up routing** — clinical pathway based on the result (surveillance / colposcopy / biopsy).

## Key design decisions

- Sophia's `Patient` class is extended (not replaced) — our `Patient` adds clinical fields on top of hers, so her queue functions work without modification.
- All clinical parameters live in `config.py` — changing a screening interval or LTFU probability requires editing only one file.
- Metrics are collected in a single dict that accumulates across all patients and days, then summarised at the end.

---
**Integration note**: Sophia's notebook is loaded with `%run`.
Our extended `Patient` class (from `patient.py`) is a superset of hers,
so all her queue functions work unchanged.

In [ ]:
import sys, random
from collections import defaultdict, deque
sys.path.insert(0, '../src')

import config as cfg
from patient import Patient
from population import sample_patient

# screening.py — eligibility checks, test assignment, result draws
from screening import (
    get_eligible_screenings,   # returns list of cancers this patient is eligible for
    run_screening_step,        # runs one full screen event (eligibility + draw)
    days_until_eligible,       # returns 0 / >0 / -1 for eligibility routing
    is_due_for_screening,
)

# followup.py — post-result clinical pathways
# run_cervical_followup chains: route → colposcopy → treatment
# run_lung_followup chains: communicate → biopsy referral → biopsy → treatment
from followup import run_cervical_followup, run_lung_followup

# metrics.py — initialize, record, and print simulation stats
# Alias our print_summary so Sophia's %run (cell below) doesn't overwrite it
from metrics import initialize_metrics, record_screening, record_exit
from metrics import print_summary as print_screening_summary

print('Core imports OK')

## Load Sophia's Arrival Functions

`%run` executes Sophia's notebook in its entirety and imports all of her functions and global variables into this notebook's namespace. This is equivalent to copy-pasting her code here, but means we always use her latest version without duplication.

After `%run`, we override only the patient-generation step (`generate_daily_arrivals`) to use our enriched `Patient` class — everything else (queue management, scheduling, ER routing) is hers unchanged.

**Namespace note**: `%run` will also load Sophia's `print_summary` function, which prints her arrivals summary. We import our own `print_summary` as `print_screening_summary` before this cell runs, so both can coexist.

In [2]:
%run "../reference/initial_model_NYP_flow_simulation (1).ipynb"

# Confirm Sophia's key functions are loaded
print('Sophia functions loaded:', [f for f in dir() if f in (
    'initialize_state', 'generate_daily_arrivals', 'process_provider_queue',
    'process_er_queue', 'release_scheduled_patients_for_today',
    'release_returning_er_patients', 'is_weekday', 'next_weekday',
)])


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
ARRIVAL / ACCESS SUMMARY
Total patients created:               3000

Created by type:
  outpatient:                         2082
  drop_in:                            918

Created by destination:
  pcp:                                1088
  gynecologist:                       692
  specialist:                         629
  er:                                 591

Seen by destination:
  pcp:                                600
  gynecologist:                       450
  specialist:                         300
  er:                                 375

Drop-ins converted to outpatients:    694
Critical ER returned next day:        1035
Noncritical ER scheduled outpatient:  792

Outpatient showups:                   8040
Outpatient no-shows:                  0

DAY 0
--------------------------------------------------
--- DAY 0 START ---
Patient 1 arrives as outpatient to pcp
Patie

## Extended Patient Generation

Sophia's `generate_daily_arrivals` creates a plain `Patient` with only the fields her queue model needs (id, type, destination). We replace it with `generate_enriched_daily_arrivals` which calls `population.sample_patient` instead.

`sample_patient` draws the full clinical profile for each patient — age, race, insurance, HPV status, smoking history, BMI, prior CIN, etc. — from NYC-calibrated demographic distributions. These fields are what the screening eligibility checks and result-draw functions read when they evaluate the patient downstream.

This is the only change we make to the arrivals layer. Once a patient is created, Sophia's queue routing functions handle them the same way they would handle a plain `Patient` object.

In [ ]:
def generate_enriched_daily_arrivals(day: int, state: dict, next_patient_id: int) -> int:
    """
    Generate DAILY_PATIENTS enriched patients for one weekday.

    Replaces Sophia's generate_daily_arrivals with population.sample_patient so
    each patient carries demographics and clinical flags from the population model.
    Queue routing logic (drop-in vs. outpatient, ER vs. PCP/GYN) mirrors Sophia's original.

    Returns the updated next_patient_id counter.
    """
    for _ in range(cfg.DAILY_PATIENTS):
        # Draw type and destination using config probability tables
        patient_type = random.choices(
            list(cfg.PATIENT_TYPE_PROBS.keys()),
            weights=list(cfg.PATIENT_TYPE_PROBS.values())
        )[0]
        destination = random.choices(
            list(cfg.DESTINATION_PROBS.keys()),
            weights=list(cfg.DESTINATION_PROBS.values())
        )[0]

        # sample_patient creates a Patient with age, race, clinical flags, etc.
        # from the population distribution — this is the enrichment over Sophia's version
        patient = sample_patient(next_patient_id, day, destination, patient_type)
        next_patient_id += 1

        # ER patients get a critical flag — critical patients return the next day
        # for follow-up instead of being routed to an outpatient queue
        if destination == 'er':
            patient.critical_status = random.random() < cfg.ER_CRITICAL_PROB

        # Append to global patient list and increment arrival counters
        state['all_patients'].append(patient)
        state['patients_created'] += 1
        state['created_by_type'][patient_type] += 1
        state['created_by_destination'][destination] += 1

        # Route into today's queue: outpatients go through scheduling,
        # ER drop-ins go directly to the ER queue
        if patient_type == 'outpatient':
            add_patient_to_today_queue(patient, state)
        elif destination == 'er':
            state['er_today'].append(patient)
        else:
            add_patient_to_today_queue(patient, state)

    return next_patient_id

print('generate_enriched_daily_arrivals defined')

## Post-Provider Screening Step

This is the **integration point** between Sophia's layer and ours. Every time a patient is seen by a provider (PCP, GYN, or Specialist), we immediately run Steps 2–5:

1. **Eligibility** — `get_eligible_screenings` checks age, cervix status, and smoking history against USPSTF criteria for each active cancer type.
2. **Screening** — `run_screening_step` assigns the test modality and draws a stochastic result. For lung, it also runs the pre-LDCT referral + scheduling pathway.
3. **Follow-up** — `run_cervical_followup` or `run_lung_followup` routes the result through the clinical decision tree (colposcopy, biopsy, treatment).

Patients who are ineligible for all active cancers are counted and skipped. Patients who exit the system mid-encounter (e.g. lost to follow-up during lung pre-LDCT) are not processed for additional cancers.

In [ ]:
def run_post_provider_screening(patient: Patient, day: int, metrics: dict) -> None:
    """
    Execute screening steps for one patient who has just been seen by a provider.
    Updates the patient object in place and writes results to metrics.

    Flow:
      1. Get the list of cancer types this patient is eligible for right now.
      2. If none → patient is not eligible for any active screening today; just count and return.
      3. For each eligible cancer → run_screening_step (lung pre-LDCT pathway runs inside).
      4. For each non-None result → route to cancer-specific follow-up function.
    """
    metrics['n_patients'] += 1

    # get_eligible_screenings filters by ACTIVE_CANCERS and applies all eligibility rules
    eligible = get_eligible_screenings(patient)

    if not eligible:
        # Patient doesn't meet criteria for any active cancer screening today
        # (wrong age, no cervix, not enough pack-years, etc.) — nothing to do
        metrics['n_unscreened'] += 1
        return

    metrics['n_eligible_any'] += 1

    for cancer in eligible:
        # If the patient exited mid-encounter (e.g. lost to follow-up in lung pre-LDCT),
        # stop processing further cancers for this visit
        if not patient.active:
            break

        # run_screening_step checks interval, assigns test, draws result, writes to patient.
        # Returns None if skipped (not yet due) or patient was lost before scan.
        # Passes metrics so lung funnel counters (referral, scheduled, completed) are populated.
        result = run_screening_step(patient, cancer, day, metrics=metrics)
        if result is None:
            continue  # not due yet, or lost before scan — no follow-up needed

        # Write the (cancer, result) pair to the metrics screening table
        record_screening(metrics, patient, cancer, result)

        # Route to the cancer-specific follow-up chain.
        # cervical: route_cervical_result → run_colposcopy → run_treatment
        # lung:     communicate → biopsy referral → biopsy → treatment
        # Other cancers are excluded by ACTIVE_CANCERS in get_eligible_screenings.
        if cancer == 'cervical':
            run_cervical_followup(patient, day, metrics)
        elif cancer == 'lung':
            run_lung_followup(patient, day, metrics)

        # If the follow-up chain marked the patient as exited, record that exit reason
        if patient.exit_reason:
            record_exit(metrics, patient.exit_reason)

print('run_post_provider_screening defined')

## Extended Provider Queue Processor

Sophia's `process_provider_queue` drains the daily queue for one provider type, marks patients as seen up to the day's capacity, and reschedules the overflow. We wrap it here to inject the screening step for every seen patient.

The ER is handled separately by Sophia's `process_er_queue` — we do not screen in the ER in the current model (no cancer screening is appropriate in an emergency setting).

In [ ]:
def process_provider_queue_with_screening(
    day: int, queue: deque, capacity: int,
    provider_name: str, state: dict, metrics: dict
) -> None:
    """
    Sophia's provider queue logic plus a screening step for every seen patient.

    Patients up to `capacity` are marked seen; the rest are rescheduled.
    For each seen patient, run_post_provider_screening immediately triggers
    Steps 2–5 (eligibility → test → result → follow-up).
    """
    seen = 0

    while queue:
        patient = queue.popleft()

        if seen < capacity:
            # Patient is seen today — increment counter and log the visit
            seen += 1
            state['seen_by_destination'][provider_name] += 1
            log_day(state, day, f'Patient {patient.patient_id} seen by {provider_name}')

            # ── Steps 2–5: screening + follow-up ────────────────────────────
            # Runs immediately after the provider visit, same day
            run_post_provider_screening(patient, day, metrics)

        else:
            # Provider is at capacity — patient must wait and come back
            patient.wait_days += 1
            state['not_seen_by_destination'][provider_name] += 1

            # Drop-ins are converted to outpatient appointments for the next weekday;
            # already-scheduled outpatients are simply pushed to the next weekday slot
            if patient.patient_type == 'drop_in':
                patient.patient_type  = 'outpatient'
                patient.scheduled_day = next_weekday(day)
                state['future_schedule'][patient.scheduled_day].append(patient)
                state['converted_dropin_to_outpatient'] += 1
            else:
                patient.scheduled_day = next_weekday(day)
                state['future_schedule'][patient.scheduled_day].append(patient)

print('process_provider_queue_with_screening defined')

## Main Daily Process (SimPy Generator)

This is the **SimPy event loop**. SimPy is a discrete-event simulation framework — rather than running a real clock, it steps through simulation time one day at a time. The `daily_process_with_screening` generator is registered with SimPy and called once per simulated day.

On each weekday, the generator:
1. Releases outpatients whose scheduled appointment falls on this day.
2. Brings back critical ER patients from the day before.
3. Generates new patient arrivals (enriched with clinical flags).
4. Drains the PCP, GYN, and Specialist queues — screening runs for each seen patient.
5. Drains the ER queue via Sophia's original processor.
6. Advances the clock by 1 day (`yield env.timeout(1)`).

Weekends are skipped entirely to match NYP's operating schedule.

In [ ]:
def daily_process_with_screening(env, state: dict, metrics: dict):
    """
    SimPy generator: drives one simulated weekday at a time until SIM_DAYS.

    Each iteration:
      1. Skip weekends — yield a 1-day timeout and continue.
      2. Release scheduled outpatients whose appointment is today.
      3. Release returning ER patients (critical patients from yesterday).
      4. Generate new enriched arrivals (replaces Sophia's plain arrivals).
      5. Process PCP, GYN, and Specialist queues with integrated screening.
      6. Process ER queue using Sophia's original ER logic (no screening in ER).
      7. Advance SimPy clock by 1 day.
    """
    import simpy
    next_patient_id = 1

    while env.now < cfg.SIM_DAYS:
        day = int(env.now)

        # Skip weekends — Sophia's is_weekday() returns False for Sat/Sun
        if not is_weekday(day):
            yield env.timeout(1)
            continue

        # Pull today's scheduled outpatients out of future_schedule into their queues
        release_scheduled_patients_for_today(day, state)
        # Move critical ER patients from yesterday back to the ER queue
        release_returning_er_patients(day, state)

        # Generate DAILY_PATIENTS enriched arrivals and route them into queues
        next_patient_id = generate_enriched_daily_arrivals(day, state, next_patient_id)

        # Process each non-ER provider queue (steps 2–5 run inside for each seen patient)
        process_provider_queue_with_screening(
            day, state['pcp_today'],        cfg.PROVIDER_CAPACITY['pcp'],
            'pcp',          state, metrics
        )
        process_provider_queue_with_screening(
            day, state['gyn_today'],        cfg.PROVIDER_CAPACITY['gynecologist'],
            'gynecologist', state, metrics
        )
        process_provider_queue_with_screening(
            day, state['specialist_today'], cfg.PROVIDER_CAPACITY['specialist'],
            'specialist',   state, metrics
        )

        # ER uses Sophia's original processor — no cancer screening in the ER for now
        process_er_queue(day, state)

        # Advance clock by one day
        yield env.timeout(1)

print('daily_process_with_screening defined')

## Run the Simulation

`run_simulation` is the single entry point for a full model run. It:
- Seeds the random number generator so runs are reproducible.
- Creates a fresh SimPy environment (the discrete-event clock).
- Initialises both state dicts (Sophia's arrivals state and our screening metrics).
- Registers `daily_process_with_screening` with SimPy and advances to `sim_days`.

The quick test below runs **1 year (365 days)** with a fixed seed. For variance analysis, call `run_simulation` in a loop with different seeds and aggregate across replications.

In [ ]:
import simpy

def run_simulation(
    sim_days: int = cfg.SIM_DAYS,
    seed:     int = cfg.RANDOM_SEED,
):
    """
    Run the full end-to-end simulation.

    Sets the random seed, creates a SimPy environment, initialises Sophia's
    arrivals state dict and our screening metrics dict, then advances the
    SimPy clock until sim_days.

    Returns
    -------
    state   : Sophia's arrivals state dict (patients_created, seen_by_destination, etc.)
    metrics : Steps 2–6 metrics dict (screened, abnormal rates, LTFU counts, etc.)
    """
    random.seed(seed)   # fix seed so results are reproducible across runs

    env     = simpy.Environment()   # SimPy clock starts at 0 (= day 0)
    state   = initialize_state()    # Sophia's arrivals state (queues, counters, schedule)
    metrics = initialize_metrics()  # our screening/follow-up metrics (from metrics.py)

    # Register the daily process as a SimPy generator — it yields timeout(1) each day
    env.process(daily_process_with_screening(env, state, metrics))
    env.run(until=sim_days)         # runs all days synchronously (no real concurrency)

    return state, metrics

print('run_simulation defined')

## Quick Test Run — 1 Year, 1 Replication

This cell executes the full simulation for 365 simulated days and prints two summaries:
1. Our **screening metrics summary** (`print_screening_summary`) — eligibility, result distributions, LTFU counts, colposcopy and treatment volumes.
2. Sophia's **arrivals summary** (`print_summary`) — patient creation, provider utilisation, scheduling and overflow stats.

After confirming the simulation runs correctly here, the visualisations below convert these numbers into charts for easier interpretation.

In [ ]:
print('Running 1-year simulation...')
state, metrics = run_simulation(sim_days=365, seed=42)
print(f"Done. Patients created: {state['patients_created']:,}")
print()

# print_screening_summary is our aliased import from metrics.py.
# We aliased it in the imports cell so Sophia's %run doesn't overwrite it.
print_screening_summary(metrics)

## Simulation Outcomes — Visualizations

The three panels below summarise what the simulation produced:

1. **Screening Volume Funnel** — of all patients seen by a provider, how many were eligible for at least one cancer screen, and how many actually received a cervical or lung screen?  This reveals the drop-off between eligibility and actual service delivery — a key performance gap to close.

2. **Cervical Result Distribution** — what mix of normal vs. abnormal results did the simulation generate? Abnormal categories (ASCUS through HSIL, HPV+) are the patients who need colposcopy follow-up.  Comparing this distribution against NYP EHR data validates the config probabilities.

3. **Loss-to-Follow-Up (LTFU) by Node** — at which clinical decision points did patients drop out?  Post-abnormal LTFU (patient never books colposcopy) and post-colposcopy LTFU (patient never completes treatment) are the two biggest care-gap levers.

In [ ]:
import matplotlib.pyplot as plt

# ── Panel 1: Screening Volume Funnel ──────────────────────────────────────────
# Each stage represents a gate the patient must pass through.
# The gap between consecutive stages is the drop-off at that point in the pathway.
stages  = ['Patients seen', 'Eligible (≥1 screen)', 'Screened — cervical', 'Screened — lung']
volumes = [
    metrics['n_patients'],
    metrics['n_eligible_any'],
    metrics['n_screened']['cervical'],
    metrics['n_screened']['lung'],
]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
colors = ['#1565C0', '#1976D2', '#42A5F5', '#90CAF9']
# Reverse both lists so the first stage appears at the top of the horizontal chart
bars = ax.barh(stages[::-1], volumes[::-1], color=colors[::-1])
ax.set_xlabel('Number of patients')
ax.set_title('Screening volume funnel\n(1-year simulation)', fontsize=11)
# Annotate each bar with the patient count
for bar, vol in zip(bars, volumes[::-1]):
    ax.text(bar.get_width() + max(volumes)*0.01,
            bar.get_y() + bar.get_height()/2,
            f'{vol:,}', va='center', fontsize=9)
ax.set_xlim(0, max(volumes) * 1.15)

# ── Panel 2: Cervical Result Distribution ─────────────────────────────────────
# Only plot categories that actually appeared in the simulation results.
# Green = normal, yellow/orange/red spectrum = increasing severity of abnormality.
ax = axes[1]
if metrics['cervical_results']:
    cat_colors = {
        'NORMAL':        '#4CAF50',
        'HPV_NEGATIVE':  '#64B5F6',
        'ASCUS':         '#FFC107',
        'LSIL':          '#FF9800',
        'ASC-H':         '#FF5722',
        'HSIL':          '#B71C1C',
        'HPV_POSITIVE':  '#1565C0',
    }
    cats   = [c for c in cat_colors if metrics['cervical_results'].get(c, 0) > 0]
    counts = [metrics['cervical_results'][c] for c in cats]
    total  = sum(counts)
    pcts   = [c / total * 100 for c in counts]
    bar_colors = [cat_colors[c] for c in cats]

    bars2 = ax.bar(cats, pcts, color=bar_colors)
    ax.set_ylabel('% of cervical screenings')
    ax.set_title('Cervical result distribution\n(simulation output)', fontsize=11)
    ax.set_ylim(0, max(pcts) * 1.3)
    for bar, pct in zip(bars2, pcts):
        ax.text(bar.get_x() + bar.get_width()/2, pct + 0.5,
                f'{pct:.1f}%', ha='center', fontsize=8)
    plt.setp(ax.get_xticklabels(), rotation=30, ha='right')
else:
    ax.text(0.5, 0.5, 'No cervical results\nin this run',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_title('Cervical result distribution')

# ── Panel 3: LTFU Breakdown by Node ──────────────────────────────────────────
# Each bar shows how many patients were lost at that specific decision point.
# Longer bars = bigger care gap = higher priority for intervention.
ax = axes[2]
nodes = ['Post-abnormal\n(no colposcopy)', 'Post-colposcopy\n(no treatment)', 'Declined screening']
ltfu  = [
    metrics['ltfu_post_abnormal'],
    metrics['ltfu_post_colposcopy'],
    metrics['ltfu_unscreened'],
]
ltfu_colors = ['#EF5350', '#B71C1C', '#BDBDBD']
bars3 = ax.barh(nodes[::-1], ltfu[::-1], color=ltfu_colors[::-1])
ax.set_xlabel('Patients lost at this node')
ax.set_title('Loss-to-follow-up breakdown\nby node', fontsize=11)
for bar, n in zip(bars3, ltfu[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(n), va='center', fontsize=9)
ax.set_xlim(0, max(ltfu + [1]) * 1.2)

plt.suptitle('Simulation summary — 1-year run', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## Lung LDCT Pathway Funnel (Simulation Output)

This funnel shows the real-world yield of the lung cancer screening pathway as modelled in the simulation.

Each row is a required clinical step. The step count drops as patients fail to clear each gate (LTFU, no referral, no scheduling, etc.). The gap between consecutive steps represents patients who were lost at that specific node — these are the highest-priority targets for care coordination interventions.

This view directly mirrors the clinical flow chart: LDCT order → scheduling → scan → result communication → biopsy (if RADS 4) → malignancy confirmation → treatment.

In [ ]:
if metrics['lung_eligible'] > 0:
    # Each step in the lung pathway is tracked as a counter in metrics.
    # We read them in clinical order to build the funnel.
    steps = [
        ('Eligible (age 50–80, ≥20 pk-yrs)',  metrics['lung_eligible']),
        ('LDCT order placed',                  metrics['lung_referral_placed']),
        ('LDCT scheduled',                     metrics['lung_ldct_scheduled']),
        ('LDCT completed',                     metrics['lung_ldct_completed']),
        ('Result communicated',                metrics['lung_result_communicated']),
        ('Biopsy referral (RADS 4)',           metrics['lung_biopsy_referral']),
        ('Biopsy scheduled',                   metrics['lung_biopsy_scheduled']),
        ('Biopsy completed',                   metrics['lung_biopsy_completed']),
        ('Malignancy confirmed',               metrics['lung_malignancy_confirmed']),
        ('Treatment given',                    metrics['lung_treatment_given']),
    ]

    # Only show steps that had at least some patients reach them;
    # otherwise later biopsy/malignancy steps clutter the chart with zeros
    steps = [(label, count) for label, count in steps if count > 0]
    labels = [s[0] for s in steps]
    counts = [s[1] for s in steps]
    n_start = max(counts[0], 1)   # denominator for % calculation

    # Color gradient: blue for LDCT steps, red for biopsy/malignancy, green for treatment
    palette = plt.cm.Blues_r(range(50, 250, max(1, 200 // len(counts))))
    if len(counts) > 5:
        palette[-2] = (0.90, 0.33, 0.32, 1.0)   # malignancy confirmed — red
        palette[-1] = (0.40, 0.73, 0.42, 1.0)   # treatment given — green

    fig, ax = plt.subplots(figsize=(11, max(4, len(steps) * 0.6)))
    bars = ax.barh(labels[::-1], counts[::-1], color=palette[::-1])
    ax.set_xlabel('Number of patients')
    ax.set_title('Lung LDCT pathway funnel — simulation output', fontsize=12)

    # Annotate each bar: absolute count and % of eligible patients
    for bar, cnt in zip(bars, counts[::-1]):
        pct = cnt / n_start * 100
        ax.text(bar.get_width() + n_start * 0.01,
                bar.get_y() + bar.get_height() / 2,
                f'{cnt:,}  ({pct:.1f}%)', va='center', fontsize=9)
    ax.set_xlim(0, n_start * 1.25)
    plt.tight_layout()
    plt.show()
else:
    print('No lung-eligible patients were seen in this simulation run.')

## Arrivals Summary (Sophia's Layer)

The cell below prints Sophia's arrivals summary from the same simulation run. This covers everything that happened *before* the screening step: how many patients were created, how many were seen by each provider type, how many were rescheduled due to capacity, and ER overflow statistics.

Use this alongside the screening summary above to understand both layers of the simulation — access bottlenecks upstream can reduce the number of patients who ever reach a screening opportunity.

In [ ]:
# Sophia's %run (cell above) loaded her own print_summary into the global namespace.
# Call it here with `state` (the arrivals dict) to display the access/queue summary.
# Our screening metrics summary was already printed above via print_screening_summary(metrics).
print_summary(state)

## Patient Event Log Trace

This section prints the full event-by-event history for a sample of individual patients.

Reading a patient trace is the best way to verify the clinical logic is flowing correctly end-to-end. Each line shows the simulated day and the event that occurred: screening, result, routing decision, colposcopy/biopsy, treatment, exit reason, etc.

If something is wrong in the model (e.g. a patient goes to colposcopy without an abnormal result, or a follow-up step fires for an ineligible patient), it will be visible here before it propagates into aggregate statistics.

In [ ]:
from metrics import print_patient_trace

# Re-import after %run may have clobbered the name — use explicit module path to be safe.
# Filter to patients who have at least one event (i.e. were seen and screened).
# print_patient_trace prints the (day, event) log for the first n patients.
screened_patients = [p for p in state['all_patients'] if p.event_log]
print_patient_trace(screened_patients, n=3)